### Limpeza e Tratamento do Dataset

In [1]:
# ======================================================
# 1. 📚 Importações e Configuração
# Esta seção importa todas as bibliotecas necessárias para a manipulação de dados
# (pandas, numpy), gerenciamento de arquivos (os), pré-processamento de dados
# (StandardScaler, train_test_split) e modelagem de regressão
# (LinearRegression, RandomForestRegressor, metrics).
# ======================================================

import pandas as pd
import numpy as np
import os 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ======================================================
# 2. 📁 Leitura do Dataset
# O dataset Hotel Booking Demand é carregado a partir dos dados brutos (raw).
# O objetivo do projeto é realizar a análise, tratamento e modelagem preditiva
# para prever a Tarifa Média Diária (adr) de uma reserva de hotel.
# ======================================================

df = pd.read_csv("data/raw/hotel_bookings.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [3]:
# ======================================================
# 3. 🧹 Limpeza Inicial dos Dados
# Esta fase foca na limpeza básica dos dados, conforme descrito no relatório.
# - **Remoção de Duplicatas:** Linhas duplicadas são eliminadas para garantir a unicidade das observações.
# - **Remoção de Não-Hóspedes:** Reservas sem hóspedes válidos (adultos, crianças ou bebês) são removidas.
# - **Tratamento de Valores Nulos (NaN):**
# - As colunas `agent` e `company` têm seus valores ausentes preenchidos com **0**.
# - Os valores ausentes da coluna `country` são preenchidos com a **moda**.
# - **Tratamento de `adr` Inválido:** Valores menores ou iguais a zero são removidos, pois a `adr` é a variável alvo e esses valores não são consistentes.
# - **Engenharia de Feature (`faixa_preco`):**
# A variável categórica `faixa_preco` é criada a partir dos quartis da `adr`, classificando os preços em *Econômica*, *Padrão*, *Premium* e *Luxo*. Essa variável será removida posteriormente para evitar *Data Leakage*.
# ======================================================

# Eliminando duplicatas

duplicated = df.duplicated().sum()
print(duplicated)
df.drop_duplicates(inplace=True)

31994


In [4]:
# Dataset sem as linhas com 0 hóspedes
df_limpo = df[(df['adults'] > 0) | (df['children'] > 0) | (df['babies'] > 0)].copy()
print(len(df))
print(len(df_limpo))

87396
87230


In [5]:
# Preenchendo os valores nulos das colunas 'company' e 'agent' com o valor '0'
df_limpo['company'] = df_limpo['company'].fillna(0)
df_limpo['agent'] = df_limpo['agent'].fillna(0)

In [6]:
## Identificando valores ausentes na coluna 'country'
df_limpo.info()
df_limpo["country"].isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 87230 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87230 non-null  object 
 1   is_canceled                     87230 non-null  int64  
 2   lead_time                       87230 non-null  int64  
 3   arrival_date_year               87230 non-null  int64  
 4   arrival_date_month              87230 non-null  object 
 5   arrival_date_week_number        87230 non-null  int64  
 6   arrival_date_day_of_month       87230 non-null  int64  
 7   stays_in_weekend_nights         87230 non-null  int64  
 8   stays_in_week_nights            87230 non-null  int64  
 9   adults                          87230 non-null  int64  
 10  children                        87226 non-null  float64
 11  babies                          87230 non-null  int64  
 12  meal                            8723

np.int64(447)

In [7]:
# Substituindo valores nulos pela moda da coluna 'country'
moda_country = df_limpo['country'].mode()[0]
df_limpo['country'] = df_limpo['country'].fillna(moda_country)

In [8]:
## Tratando coluna 'adr' com valores negativos e iguais a 0

adr_negativo = df[df['adr'] < 0]
adr_zero = df[df['adr'] == 0]

# Excluindo linhas com valores menores ou iguais a zero.
df_limpo = df_limpo[(df_limpo['adr']) > 0].copy()
df_limpo.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85582.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000,85586.000000
mean,0.278550,80.719078,2016.215584,26.798542,15.824118,1.017024,2.652677,1.881394,0.140064,0.010761,0.034129,0.028965,0.168719,0.265300,81.532108,10.752027,0.741698,108.564183,0.084757,0.700009
std,0.448288,85.933048,0.684357,13.636826,8.836808,1.026187,2.031387,0.501666,0.458054,0.113529,0.181563,0.364826,1.675545,0.702561,110.072130,53.618644,10.000718,53.373911,0.282396,0.830951
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.260000,0.000000,0.000000
25%,0.000000,12.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,74.500000,0.000000,0.000000
50%,0.000000,50.000000,2016.000000,27.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,99.000000,0.000000,0.000000
75%,1.000000,126.000000,2017.000000,37.000000,24.000000,2.000000,4.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,240.000000,0.000000,0.000000,135.000000,0.000000,1.000000
max,1.000000,709.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,4.000000,10.000000,10.000000,1.000000,26.000000,72.000000,18.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [9]:
### Engenharia de Feature

## Disitribuição estatística da coluna 'adr'
descritivo_adr = df_limpo['adr'].describe()

q1 = descritivo_adr['25%']  # 74.0
q2 = descritivo_adr['50%']  # 99.0
q3 = descritivo_adr['75%']  # 135.0

bins = [-float('inf'), q1, q2, q3, float('inf')]
labels = ['Econômica', 'Padrão', 'Premium', 'Luxo']

In [10]:
## Criando a coluna 'faixa_preco'
df_limpo['faixa_preco'] = pd.cut(df_limpo['adr'], bins=bins, labels=labels, right=False)

In [11]:
# Salvando dataset tratado
df_limpo.to_csv(os.path.join("data", "processed", "hotel_bookings_cleaned.csv"), index=False)


In [12]:
# ======================================================
# 4. 📊 Tratamento de Outliers (Método IQR)
# Valores extremos nas colunas `adr`, `lead_time`, `stays_in_week_nights` e `stays_in_weekend_nights` são tratados utilizando o método do **Intervalo Interquartil (IQR)**.
# Observações fora dos limites $Q1 - 1.5 \cdot IQR$ e $Q3 + 1.5 \cdot IQR$ são removidas.
# Esse procedimento reduz o impacto de valores extremos que poderiam distorcer o treinamento do modelo, seja por inconsistências reais ou erros de medição.
# ======================================================

colunas_outliers = [
    'adr',
    'lead_time',
    'stays_in_week_nights',
    'stays_in_weekend_nights'
]

for col in colunas_outliers:
    Q1 = df_limpo[col].quantile(0.25)
    Q3 = df_limpo[col].quantile(0.75)
    IQR = Q3 - Q1

    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    df_limpo = df_limpo[
        (df_limpo[col] >= limite_inf) &
        (df_limpo[col] <= limite_sup)
    ]

print("Tamanho após remoção de outliers:", len(df_limpo))

Tamanho após remoção de outliers: 78088


In [13]:
# ======================================================
# 5. 🔍 Seleção de Colunas
# Nesta etapa é realizada a definição final das colunas que permanecerão no dataset após o tratamento, priorizando variáveis relevantes para o problema de regressão e reduzindo ruídos desnecessários.
# ======================================================

colunas_mantidas = [
    'adults',
    'children',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'meal',
    'required_car_parking_spaces',
    'reserved_room_type',
    'lead_time',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month',
    'market_segment',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'adr',
    'total_of_special_requests',
    'reservation_status',
    'faixa_preco'
]

df_limpo = df_limpo[colunas_mantidas].copy()

df_limpo.info()


<class 'pandas.core.frame.DataFrame'>
Index: 78088 entries, 2 to 119388
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   adults                          78088 non-null  int64   
 1   children                        78084 non-null  float64 
 2   stays_in_weekend_nights         78088 non-null  int64   
 3   stays_in_week_nights            78088 non-null  int64   
 4   meal                            78088 non-null  object  
 5   required_car_parking_spaces     78088 non-null  int64   
 6   reserved_room_type              78088 non-null  object  
 7   lead_time                       78088 non-null  int64   
 8   arrival_date_year               78088 non-null  int64   
 9   arrival_date_month              78088 non-null  object  
 10  arrival_date_day_of_month       78088 non-null  int64   
 11  market_segment                  78088 non-null  object  
 12  is_repeated_guest     

In [14]:
# ======================================================
# 6. 🛑 Separação X (Features) e y (Alvo) – Prevenção de Data Leakage
# A variável alvo $y$ é definida como a Tarifa Média Diária (`adr`).
# As variáveis explicativas $X$ são selecionadas removendo colunas que causariam **Data Leakage**, pois contêm informações futuras ou derivadas da variável alvo:
# - `adr`: variável alvo.
# - `reservation_status`: status final da reserva (informação futura).
# - `faixa_preco`: variável derivada diretamente da `adr`.
# ======================================================

# Variável alvo
y = df_limpo['adr']

# Removendo colunas que causam data leakage
X = df_limpo.drop([
    'adr',                        # variável resposta
    'reservation_status',         # informação futura
    'faixa_preco'                 # derivada da variável alvo
], axis=1)

print("Colunas utilizadas no modelo:")
print(X.columns)


Colunas utilizadas no modelo:
Index(['adults', 'children', 'stays_in_weekend_nights', 'stays_in_week_nights',
       'meal', 'required_car_parking_spaces', 'reserved_room_type',
       'lead_time', 'arrival_date_year', 'arrival_date_month',
       'arrival_date_day_of_month', 'market_segment', 'is_repeated_guest',
       'previous_cancellations', 'previous_bookings_not_canceled',
       'total_of_special_requests'],
      dtype='object')


In [15]:
# ======================================================
# 7. 🛠️ Imputação de NaN
# Valores ausentes remanescentes nas colunas numéricas (como `children`) são preenchidos com a **mediana**.
# Essa estratégia preserva a distribuição dos dados e evita a redução do tamanho do dataset após o tratamento de outliers.
# ======================================================

# Verificando valores ausentes após seleção das features
X.isnull().sum().sort_values(ascending=False).head(10)

# Colunas numéricas
colunas_numericas = X.select_dtypes(include=['int64', 'float64']).columns

# Preenchendo NaN com a mediana
for col in colunas_numericas:
    mediana = X[col].median()
    X[col] = X[col].fillna(mediana)

# ======================================================
# 8. 🔄 One-Hot Encoding
# As variáveis categóricas restantes (como `meal`, `arrival_date_month`, `market_segment` e `reserved_room_type`) são convertidas em variáveis binárias utilizando **One-Hot Encoding**, permitindo seu uso em modelos de regressão.
# ======================================================
X = pd.get_dummies(X, drop_first=True)

# ======================================================
# 9. ⚖️ Normalização (StandardScaler)
# As variáveis numéricas — incluindo as *dummies* — são normalizadas utilizando o `StandardScaler`.
# Esse pré-processamento garante que todas as features tenham média zero e desvio padrão unitário, evitando que variáveis em escalas maiores dominem o treinamento do modelo.
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ======================================================
# 10. 📝 Train/Test Split
# O conjunto de dados é dividido em treino e teste na proporção **80/20**, garantindo que a avaliação do modelo seja realizada em dados não vistos durante o treinamento.
# ======================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# ======================================================
# 11. 🧠 Treinamento do Modelo
# O modelo de **Regressão Linear** é instanciado e treinado utilizando o conjunto de dados de treinamento.
# O objetivo é modelar a relação entre as variáveis explicativas e a Tarifa Média Diária (`adr`).
# ======================================================

modelo = LinearRegression()
modelo.fit(X_train, y_train)
preds = modelo.predict(X_test)

# ======================================================
# 12. ✅ Avaliação do Modelo
# O desempenho do modelo é avaliado no conjunto de teste utilizando as seguintes métricas:
# - **MAE (Erro Absoluto Médio):** Mede o erro médio absoluto das previsões.
# - **RMSE (Raiz do Erro Quadrático Médio):** Penaliza erros maiores, fornecendo uma noção da magnitude típica do erro.
# - **R² (Coeficiente de Determinação):** Indica a proporção da variância da variável alvo explicada pelo modelo.
# Os resultados indicam um desempenho satisfatório e um pipeline alinhado às boas práticas de ciência de dados.
# ======================================================
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")



MAE: 22.61
RMSE: 28.56
R²: 0.57


In [16]:
X.columns

Index(['adults', 'children', 'stays_in_weekend_nights', 'stays_in_week_nights',
       'required_car_parking_spaces', 'lead_time', 'arrival_date_year',
       'arrival_date_day_of_month', 'is_repeated_guest',
       'previous_cancellations', 'previous_bookings_not_canceled',
       'total_of_special_requests', 'meal_FB', 'meal_HB', 'meal_SC',
       'meal_Undefined', 'reserved_room_type_B', 'reserved_room_type_C',
       'reserved_room_type_D', 'reserved_room_type_E', 'reserved_room_type_F',
       'reserved_room_type_G', 'reserved_room_type_H', 'reserved_room_type_L',
       'arrival_date_month_August', 'arrival_date_month_December',
       'arrival_date_month_February', 'arrival_date_month_January',
       'arrival_date_month_July', 'arrival_date_month_June',
       'arrival_date_month_March', 'arrival_date_month_May',
       'arrival_date_month_November', 'arrival_date_month_October',
       'arrival_date_month_September', 'market_segment_Complementary',
       'market_segment_Corpo

In [17]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78088 entries, 2 to 119388
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   adults                          78088 non-null  int64   
 1   children                        78084 non-null  float64 
 2   stays_in_weekend_nights         78088 non-null  int64   
 3   stays_in_week_nights            78088 non-null  int64   
 4   meal                            78088 non-null  object  
 5   required_car_parking_spaces     78088 non-null  int64   
 6   reserved_room_type              78088 non-null  object  
 7   lead_time                       78088 non-null  int64   
 8   arrival_date_year               78088 non-null  int64   
 9   arrival_date_month              78088 non-null  object  
 10  arrival_date_day_of_month       78088 non-null  int64   
 11  market_segment                  78088 non-null  object  
 12  is_repeated_guest     

In [18]:
# ======================================================
# 13. SALVANDO DATASET PROCESSADO
# ======================================================
df_limpo.to_csv(
    "data/processed/hotel_bookings_processed.csv",
    index=False
)

print("Dataset processado salvo com sucesso!")
print("Colunas finais:", df_limpo.columns)


Dataset processado salvo com sucesso!
Colunas finais: Index(['adults', 'children', 'stays_in_weekend_nights', 'stays_in_week_nights',
       'meal', 'required_car_parking_spaces', 'reserved_room_type',
       'lead_time', 'arrival_date_year', 'arrival_date_month',
       'arrival_date_day_of_month', 'market_segment', 'is_repeated_guest',
       'previous_cancellations', 'previous_bookings_not_canceled', 'adr',
       'total_of_special_requests', 'reservation_status', 'faixa_preco'],
      dtype='object')
